In [ ]:
import json
import os

import h5py
import numpy as np
import robomimic
import robomimic.utils.file_utils as FileUtils

# the dataset registry can be found at robomimic/__init__.py
from robomimic import DATASET_REGISTRY

In [2]:
from pathlib import Path

WS_DIR = str(Path(os.getcwd()).parent / "data")

In [3]:
DATASET_REGISTRY

{'lift': {'ph': {'raw': {'url': 'http://downloads.cs.stanford.edu/downloads/rt_benchmark/lift/ph/demo.hdf5',
    'horizon': 400},
   'low_dim': {'url': 'http://downloads.cs.stanford.edu/downloads/rt_benchmark/lift/ph/low_dim.hdf5',
    'horizon': 400},
   'image': {'url': 'http://downloads.cs.stanford.edu/downloads/rt_benchmark/lift/ph/image.hdf5',
    'horizon': 400}},
  'mh': {'raw': {'url': 'http://downloads.cs.stanford.edu/downloads/rt_benchmark/lift/mh/demo.hdf5',
    'horizon': 500},
   'low_dim': {'url': 'http://downloads.cs.stanford.edu/downloads/rt_benchmark/lift/mh/low_dim.hdf5',
    'horizon': 500},
   'image': {'url': 'http://downloads.cs.stanford.edu/downloads/rt_benchmark/lift/mh/image.hdf5',
    'horizon': 500}},
  'mg': {'raw': {'url': 'http://downloads.cs.stanford.edu/downloads/rt_benchmark/lift/mg/demo.hdf5',
    'horizon': 400},
   'low_dim_sparse': {'url': 'http://downloads.cs.stanford.edu/downloads/rt_benchmark/lift/mg/low_dim_sparse.hdf5',
    'horizon': 400},
   

In [6]:
# # set download folder and make it
download_folder = WS_DIR + "/robomimic_data/"
# os.makedirs(download_folder, exist_ok=True)

# # download the dataset
# task = "lift"
# dataset_type = "ph"
# hdf5_type = "low_dim"
# FileUtils.download_url(
#     url=DATASET_REGISTRY[task][dataset_type][hdf5_type]["url"],
#     download_dir=download_folder,
# )



In [7]:
download_folder

'/home/thankgod/2025/msc_project/src/data/robomimic_data/'

In [8]:
# enforce that the dataset exists
dataset_path = os.path.join(download_folder, "low_dim.hdf5")
assert os.path.exists(dataset_path)

In [9]:
from rich import print

In [10]:
dataset_path2 = Path(os.getcwd()).parent / 'data/lift/ph/image_abs.hdf5'
dataset_path2

PosixPath('/home/thankgod/2025/msc_project/src/data/lift/ph/image_abs.hdf5')

In [11]:
env_meta = FileUtils.get_env_metadata_from_dataset(
    str(dataset_path)
    )
print(env_meta)

{
    'env_name': 'Lift',
    'type': 1,
    'env_kwargs': {
        'has_renderer': False,
        'has_offscreen_renderer': False,
        'ignore_done': True,
        'use_object_obs': True,
        'use_camera_obs': False,
        'control_freq': 20,
        'controller_configs': {
            'type': 'OSC_POSE',
            'input_max': 1,
            'input_min': -1,
            'output_max': [0.05, 0.05, 0.05, 0.5, 0.5, 0.5],
            'output_min': [-0.05, -0.05, -0.05, -0.5, -0.5, -0.5],
            'kp': 150,
            'damping': 1,
            'impedance_mode': 'fixed',
            'kp_limits': [0, 300],
            'damping_limits': [0, 10],
            'position_limits': None,
            'orientation_limits': None,
            'uncouple_pos_ori': True,
            'control_delta': True,
            'interpolation': None,
            'ramp_ratio': 0.2
        },
        'robots': ['Panda'],
        'camera_depths': False,
        'camera_heights': 84,
        'camera_widths': 84,
        'reward_shaping': False
    }
}

In [12]:
env_meta['env_kwargs']['use_camera_obs'] = True
env_meta['env_kwargs']['has_offscreen_renderer'] = True

In [13]:
print(env_meta)

{
    'env_name': 'Lift',
    'type': 1,
    'env_kwargs': {
        'has_renderer': False,
        'has_offscreen_renderer': True,
        'ignore_done': True,
        'use_object_obs': True,
        'use_camera_obs': True,
        'control_freq': 20,
        'controller_configs': {
            'type': 'OSC_POSE',
            'input_max': 1,
            'input_min': -1,
            'output_max': [0.05, 0.05, 0.05, 0.5, 0.5, 0.5],
            'output_min': [-0.05, -0.05, -0.05, -0.5, -0.5, -0.5],
            'kp': 150,
            'damping': 1,
            'impedance_mode': 'fixed',
            'kp_limits': [0, 300],
            'damping_limits': [0, 10],
            'position_limits': None,
            'orientation_limits': None,
            'uncouple_pos_ori': True,
            'control_delta': True,
            'interpolation': None,
            'ramp_ratio': 0.2
        },
        'robots': ['Panda'],
        'camera_depths': False,
        'camera_heights': 84,
        'camera_widths': 84,
        'reward_shaping': False
    }
}

In [14]:
import robomimic.utils.env_utils as EnvUtils


In [16]:
# env = EnvUtils.create_env_from_metadata(
#     env_meta=env_meta,
#     render=False,
#     render_offscreen=True,
#     use_image_obs=False,
# )

# env.reset()

In [26]:
import os
import sys

src_parent = os.path.abspath(os.path.join(os.path.dirname(os.getcwd())))
sys.path.append(src_parent)

# import scripts
# from scripts.image_dataset import PushTImageDataset, normalize_data, unnormalize_data
# from scripts.models import ConditionalUnet1D, VisionEncoder
# from scripts.pusht_image_env import PushTImageEnv
# from scripts.training_Inference import UnetTrainer

In [ ]:
import collections
import importlib
import math
from pathlib import Path
from typing import Callable, Dict, Optional, Sequence, Tuple, Union

import torch.nn as nn
import torchvision
from diffusers.optimization import get_scheduler
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
from diffusers.training_utils import EMAModel

#from rich import print
from tqdm.auto import tqdm

In [28]:
import hydra


In [29]:
from omegaconf import OmegaConf
from torch.utils.data import DataLoader

In [31]:
# Load the config file
#cfg = OmegaConf.load('../pushT_config.yaml')
cfg = OmegaConf.load('../lift_config.yaml')
#cfg = OmegaConf.load('../image_lift_ph_diffusion_policy_cnn.yaml')



#print(cfg)
#dataset: BaseImageDataset
dataset = hydra.utils.instantiate(cfg.task.dataset)
#assert isinstance(dataset, BaseImageDataset)

Loading image data: 100%|██████████| 19332/19332 [00:06<00:00, 2810.95it/s]


In [32]:


# train dataset
train_dataloader = DataLoader(dataset, **cfg.dataloader)
normalizer = dataset.get_normalizer()

# configure validation dataset
val_dataset = dataset.get_validation_dataset()
val_dataloader = DataLoader(val_dataset, **cfg.val_dataloader)


In [33]:
model = hydra.utils.instantiate(cfg.policy)

optimizer = hydra.utils.instantiate(
            cfg.optimizer, params=model.parameters())


In [34]:
from diffusion_policy.env_runner.pusht_image_runner import PushTImageRunner

pygame 2.2.0 (SDL 2.26.5, Python 3.9.18)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [35]:
env_runner = hydra.utils.instantiate(
    cfg.task.env_runner,
    output_dir=None)

Created environment with name Lift
Action size is 7


KeyboardInterrupt: 